# 68 Preprocessing: what representation of the FFT actually encodes object position?

Every question here is answered by **one metric**, so the answers are comparable:

> **rho** = Spearman correlation between *signal distance* and *physical distance*, over all
> pairs of samples, holding the speaker fixed.

Higher rho = moving the cube reliably moves the signal. This is the property a model needs; a
representation that scores rho~0 cannot be learned from no matter how expressive the decoder is.

**Why "holding the speaker fixed" is not optional.** Speaker is by far the largest source of
variance in this dataset. Notebook 62 measured 65.6% of variance explained by speaker vs 20.3% by
position and concluded "the gastronorm capture simply does not encode object position" -- but it
computed that with `normalize_mode="std"` (a single global scalar divide) and never passed
`speaker_mean=`, so speaker was left fully intact. Its own decisive test (within-speaker position
lift) returned an empty DataFrame and the conclusion was copied from an unfulfilled `if` clause.
Notebook 69 repairs that analysis. Here we sidestep it by comparing only samples that share a
speaker.

## The setup, and why it matters for what follows

- Chirp: **50 -> 1000 Hz linear sweep, 1.0 s**, played once. Capture is **1.3 s @ 2500 fps**
  (`n_frames=3250`), so ~23% of the window has no excitation in it.
- FFT: one `rfft` over the whole 1.3 s capture -> 1235 bins at **0.77 Hz** resolution,
  cropped to [50, 1000] Hz. Stored `complex64`, shape `(1, 100, 1235, 2)` = (batch, laser, freq, xy).
- Each frequency is therefore excited for only **~1.05 ms** and never revisited. There is no
  repetition to average over, so every bin's estimate is single-shot.

A note on a tempting-but-wrong idea: a **spectrogram does not help here**. Because the system is
linear and the chirp covers the whole band, the global FFT `Y(f) = H(f) X(f)` is already a valid
estimator of the transfer function `H(f)`, and resonances are time-invariant, so time is a nuisance
axis -- exactly as you'd expect. The problem is not the transform, it is that there is only ~1 ms of
excitation per bin and nothing to average. That is fixed by **capturing repeated sweeps**, not by
re-transforming the data you already have.

## Config

In [1]:
REPO = '/home/ethantu/workspace/good-vibrations'
EXP = f'{REPO}/experiments/31_07_2026_gastronorm_exp1'

# Probe set: purple-cube is the only layout with a dense single-object position sweep
# (70 positions on a 7x10 raster), which is what makes a distance-vs-distance correlation readable.
LAYOUT = 'purple-cube'
SPEAKER = 1          # hold speaker fixed; the cross-speaker question is section 2
N_JOBS = 8

In [2]:
import sys, json, itertools
from pathlib import Path
from collections import defaultdict
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
sys.path.insert(0, f'{REPO}/src')

MDS = sorted(Path(f'{EXP}/mds').glob('*/metadata.jsonl'))[0].parent
INDEX = [json.loads(l) for l in (MDS / 'metadata.jsonl').read_text().strip().splitlines() if l]
SAMPLES = Path(f'{EXP}/samples')

def meta(sid):
    p = SAMPLES / sid / 'metadata.jsonl'
    return {k: v for d in (json.loads(l) for l in p.read_text().splitlines() if l) for k, v in d.items()}

def fft(sid):
    """(100, 1235, 2) complex64 -- (laser, freq, xy)."""
    return np.load(SAMPLES / sid / 'vibration' / '04_fft.npz')['fft'][0]

FREQS = np.load(SAMPLES / INDEX[0]['sample_id'] / 'vibration' / '04_fft.npz')['freqs']
print(f'{len(INDEX)} samples | {len(FREQS)} freq bins, {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz, df={FREQS[1]-FREQS[0]:.3f} Hz')

3007 samples | 1235 freq bins, 50.8-1000.0 Hz, df=0.769 Hz


## Load the probe set

One sample per position, speaker held fixed. `com` is the object's own centre of mass in the raw
overhead image -- **not** the `downsampled_com` in the MDS metadata, which for a multi-object scene
is the *average* over objects, i.e. the midpoint between two cubes, a point where there is usually
no object at all. For this single-object layout they agree, but the habit matters later.

In [3]:
rows = sorted([r for r in INDEX if r['layout'] == LAYOUT and r['speaker'] == SPEAKER],
              key=lambda r: r['position_id'])
print(f'{len(rows)} positions, speaker {SPEAKER}')

F, COM = [], []
for r in rows:
    F.append(fft(r['sample_id']))
    COM.append(np.asarray(meta(r['sample_id'])['coms'][0][0], dtype=float))
F = np.stack(F)                 # (N, 100, 1235, 2) complex
COM = np.stack(COM)             # (N, 2) pixels
N = len(F)
MAG, PHASE = np.abs(F), np.angle(F)
print(f'F {F.shape} {F.dtype} | COM range x[{COM[:,0].min():.0f},{COM[:,0].max():.0f}] y[{COM[:,1].min():.0f},{COM[:,1].max():.0f}]')

70 positions, speaker 1
F (70, 100, 1235, 2) complex64 | COM range x[214,830] y[230,1142]


## (e) How do we measure FFT similarity?

The choice matters more than it looks. Three candidates:

- **cosine** on the flattened vector -- scale-invariant, so a laser that happens to be brighter in
  one recording does not dominate. This is the default here.
- **euclidean** -- sensitive to overall gain, which is a nuisance factor (playback level, laser
  alignment) rather than a property of the object.
- **correlation** -- cosine after centering, i.e. it removes the mean spectrum shared by all samples.
  Since ~all of these spectra share the box's own resonances, centering removes a large common mode.

We report cosine and correlation; the metric below is a *rank* correlation, so any monotone
re-scaling of the distance leaves rho unchanged.

In [4]:
IU = np.triu_indices(N, 1)
PHYS = np.linalg.norm(COM[:, None, :] - COM[None, :, :], axis=-1)[IU]   # pixel distance between positions

def rho(X, name='', metric='cosine', verbose=True):
    """Spearman(signal distance, physical distance) over all pairs. X is (N, D) real."""
    X = X.reshape(N, -1).astype(np.float64)
    if metric == 'correlation': X = X - X.mean(0, keepdims=True)
    Xn = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)
    d = 1 - (Xn @ Xn.T)[IU]
    r, p = spearmanr(d, PHYS)
    if verbose: print(f'  {name:44s} rho={r:+.4f}  p={p:.1e}')
    return r

## (f) Do we WANT nearby positions to have similar FFTs?

Yes -- and this is worth being explicit about, because the opposite intuition ("we want them
different so we can tell them apart") is the wrong frame.

Discriminability is not the constraint; **learnability** is. A model must interpolate. If the
position -> FFT map were maximally different for nearby positions, it would be a hash: perfectly
discriminative, and impossible to generalize from, because nothing about a training position tells
you anything about a nearby test position. Every unseen position would be a fresh lookup.

So we want the map to be **smooth and injective**: nearby positions -> nearby (but distinguishable)
FFTs, far positions -> far FFTs. That is exactly what a positive `rho` measures. rho ~ 0 means the
map is a hash and the held-out-position generalization we care about is impossible.

Note this is measurable *without training anything*, and it upper-bounds what any architecture can do.

## (c) Log, and the other magnitude transforms

Resonance peaks span orders of magnitude, so a linear magnitude is dominated by the few loudest
bins. `log1p` compresses that and gives the anti-resonances -- the *dips*, which notebook 57 found
carry more position information than the peaks (dips alone rho=-0.523 vs peaks) -- comparable weight
to the peaks.

In [5]:
scale = MAG.std()
variants = {
    'magnitude (current default)': MAG,
    'log1p(magnitude)':            np.log1p(MAG / scale),
    'log10(magnitude + eps)':      np.log10(MAG + 1e-8),
    'sqrt(magnitude)':             np.sqrt(MAG),
    'power (magnitude^2)':         MAG ** 2,
}
print('metric = cosine')
res_mag = {k: rho(v, k) for k, v in variants.items()}
print('\nmetric = correlation (common box spectrum removed)')
res_corr = {k: rho(v, k, metric='correlation') for k, v in variants.items()}

metric = cosine
  magnitude (current default)                  rho=+0.3519  p=2.5e-71
  log1p(magnitude)                             rho=+0.3961  p=1.6e-91


  log10(magnitude + eps)                       rho=+0.2278  p=8.6e-30
  sqrt(magnitude)                              rho=+0.3830  p=3.2e-85
  power (magnitude^2)                          rho=+0.2836  p=6.6e-46

metric = correlation (common box spectrum removed)


  magnitude (current default)                  rho=+0.3282  p=9.3e-62
  log1p(magnitude)                             rho=+0.3861  p=1.0e-86


  log10(magnitude + eps)                       rho=+0.3662  p=1.6e-77
  sqrt(magnitude)                              rho=+0.3807  p=3.7e-84


  power (magnitude^2)                          rho=+0.2086  p=3.8e-25


## (a/b) Phase: between samples, between speakers, between lasers

The stored phase is not directly usable, but the reason is specific and fixable.

Each recording has an **unknown time origin** -- camera/audio trigger jitter of order ~1 ms. A time
shift `tau` multiplies the spectrum by `exp(-2*pi*i*f*tau)`, i.e. it adds a **linear ramp** in
frequency to the phase. At 1000 Hz, 1 ms of jitter is a full cycle, so the raw phase at high
frequency is effectively randomized between samples. That is why notebook 57 found phase made things
worse (rho -0.526 -> -0.136) and concluded it was unusable.

But the ramp is a *gauge*, not noise. Differencing adjacent frequency bins cancels it:

    dphi(f) = phi(f + df) - phi(f)

and `dphi` is (up to a constant) the **group delay**, a physically meaningful quantity that is
invariant to the trigger jitter. The cell below measures whether that recovers consistency.

**Encoding.** Phase is circular: -pi and +pi are the same angle but maximally distant as raw
numbers, so a model reading raw angles sees a false discontinuity. Encode as `(cos, sin)` instead.
We test both.

In [6]:
# Same physical position, 8 different speakers -- ask how reproducible phase is at all.
by_pos = defaultdict(list)
for r in INDEX:
    if r['layout'] == LAYOUT: by_pos[r['position_id']].append(r)
pid = sorted(by_pos)[0]
grp = sorted(by_pos[pid], key=lambda r: r['speaker'])
Fs = np.stack([fft(r['sample_id']) for r in grp])            # (8, 100, 1235, 2)
ph = np.angle(Fs)

def R(angles, axis=0):
    """Circular mean resultant length: 0 = uniform/random, 1 = identical."""
    return np.abs(np.exp(1j * angles).mean(axis=axis))

dphi = np.angle(np.exp(1j * (ph[:, :, 1:, :] - ph[:, :, :-1, :])))
print(f'position {pid}, {len(grp)} speakers')
print(f'  raw phase          R = {R(ph).mean():.4f}   <- near 0 => randomized by trigger jitter')
print(f'  group delay dphi   R = {R(dphi).mean():.4f}   <- the ramp cancels; this is reproducible')

# (b3) across lasers, within one sample: is there a shared per-sample ramp?
p0 = ph[0]                                                    # (100, 1235, 2)
print(f'\nwithin one sample, across the 100 lasers:')
print(f'  raw phase          R = {R(p0, axis=0).mean():.4f}')
print(f'  group delay        R = {R(np.angle(np.exp(1j*(p0[:,1:,:]-p0[:,:-1,:]))), axis=0).mean():.4f}')

position 6, 8 speakers
  raw phase          R = 0.4106   <- near 0 => randomized by trigger jitter
  group delay dphi   R = 0.7584   <- the ramp cancels; this is reproducible

within one sample, across the 100 lasers:
  raw phase          R = 0.2187
  group delay        R = 0.7891


In [7]:
# Does phase carry POSITION information, and in which encoding?
PH_D = np.angle(np.exp(1j * (PHASE[:, :, 1:, :] - PHASE[:, :, :-1, :])))    # (N,100,1234,2) group delay
LOGM = np.log1p(MAG / scale)

print('phase alone')
rho(np.concatenate([np.cos(PHASE).reshape(N,-1), np.sin(PHASE).reshape(N,-1)], 1), 'cos/sin RAW phase')
rho(np.concatenate([np.cos(PH_D).reshape(N,-1),  np.sin(PH_D).reshape(N,-1)],  1), 'cos/sin group delay')

# phase is only meaningful where there is energy; weight it by magnitude
W = MAG[:, :, 1:, :]
rho(np.concatenate([(np.cos(PH_D)*W).reshape(N,-1), (np.sin(PH_D)*W).reshape(N,-1)], 1),
    'magnitude-weighted cos/sin group delay')

print('\ncombined with log-magnitude at varying phase weight')
Ln = LOGM.reshape(N,-1); Ln = Ln / np.linalg.norm(Ln, axis=1, keepdims=True)
P  = np.concatenate([np.cos(PH_D).reshape(N,-1), np.sin(PH_D).reshape(N,-1)], 1)
Pn = P / np.linalg.norm(P, axis=1, keepdims=True)
rho(Ln, 'log-magnitude alone (reference)')
for w in [0.1, 0.25, 0.5, 1.0]:
    rho(np.concatenate([Ln, w * Pn], 1), f'log-magnitude + {w} * cos/sin group delay')

phase alone


  cos/sin RAW phase                            rho=+0.0207  p=3.1e-01


  cos/sin group delay                          rho=+0.0927  p=5.0e-06


  magnitude-weighted cos/sin group delay       rho=+0.3185  p=4.8e-58

combined with log-magnitude at varying phase weight


  log-magnitude alone (reference)              rho=+0.3961  p=1.6e-91


  log-magnitude + 0.1 * cos/sin group delay    rho=+0.3932  p=4.0e-90


  log-magnitude + 0.25 * cos/sin group delay   rho=+0.3749  p=1.8e-81


  log-magnitude + 0.5 * cos/sin group delay    rho=+0.3005  p=1.4e-51


  log-magnitude + 1.0 * cos/sin group delay    rho=+0.1712  p=2.4e-17


## (d) The empty box as a baseline

`empty-box` is the transfer function of the box with nothing in it. Dividing it out (or subtracting,
in log space) should remove the box's own resonances and leave only the *perturbation* caused by the
object -- which is the part that depends on position.

There are 4 empty-box positions (~31 samples). Use the **same speaker**, since the speaker term is
multiplicative in the same place.

In [8]:
empty_rows = [r for r in INDEX if r['layout'] == 'empty-box' and r['speaker'] == SPEAKER]
print(f'{len(empty_rows)} empty-box samples at speaker {SPEAKER}')
E = np.stack([np.abs(fft(r['sample_id'])) for r in empty_rows]).mean(0)   # (100,1235,2) reference

rho(LOGM, 'log-magnitude (reference)')
rho(MAG / np.clip(E, 1e-12, None),               'magnitude / empty-box')
rho(np.log1p(MAG/scale) - np.log1p(E/scale),     'log-magnitude - log empty-box')
rho(np.log10(MAG + 1e-8) - np.log10(E + 1e-8),   'log10 ratio to empty-box')

3 empty-box samples at speaker 1
  log-magnitude (reference)                    rho=+0.3961  p=1.6e-91


  magnitude / empty-box                        rho=+0.2361  p=6.1e-32
  log-magnitude - log empty-box                rho=+0.4860  p=2.2e-143


  log10 ratio to empty-box                     rho=+0.4516  p=1.1e-121


0.45162019184090507

## (g) Is the position -> FFT map unique (injective)?

If two well-separated positions produce near-identical spectra, no model can tell them apart --
that is an irreducible error floor, not a modelling failure.

Notebook 57 found exactly this on the metal box: a pair 364 units apart on opposite sides scored as
anomalously similar under 7 of 8 speakers, which it attributed to a **box mode-shape symmetry**
(mirror-image positions excite the same modes equally). Here we look for the same on gastronorm: the
pairs that are far apart physically but close in signal.

In [9]:
X = LOGM.reshape(N, -1); X = X / np.linalg.norm(X, axis=1, keepdims=True)
SIG = (1 - X @ X.T)[IU]
pi_, pj_ = IU

order = np.argsort(SIG / np.clip(PHYS, 1e-9, None))   # low signal-distance per unit physical distance
print('most degenerate pairs (far apart, nearly identical signal):')
for k in order[:10]:
    print(f'  pos {rows[pi_[k]]["position_id"]:>3d} <-> {rows[pj_[k]]["position_id"]:>3d} | '
          f'physical {PHYS[k]:7.1f} px | signal {SIG[k]:.4f}')

fig = go.Figure(go.Scattergl(x=PHYS, y=SIG, mode='markers',
                             marker=dict(size=4, opacity=.45, color='#4C78A8'),
                             text=[f'{rows[a]["position_id"]} <-> {rows[b]["position_id"]}' for a, b in zip(pi_, pj_)],
                             hovertemplate='%{text}<br>phys %{x:.0f} px<br>sig %{y:.4f}<extra></extra>'))
fig.update_layout(title=f'Signal vs physical distance (log-magnitude, speaker {SPEAKER}) | rho={spearmanr(SIG,PHYS)[0]:+.3f}',
                  xaxis_title='physical distance (px)', yaxis_title='signal distance (1 - cosine)',
                  template='plotly_white', height=460, width=760)
fig.show()

most degenerate pairs (far apart, nearly identical signal):
  pos  34 <->  63 | physical   677.4 px | signal 0.0052
  pos  18 <->  67 | physical   855.3 px | signal 0.0068
  pos  24 <->  34 | physical   598.4 px | signal 0.0047
  pos  24 <->  35 | physical   688.5 px | signal 0.0056
  pos  35 <->  68 | physical   797.7 px | signal 0.0066
  pos  34 <->  68 | physical   720.7 px | signal 0.0060
  pos  34 <->  43 | physical   626.7 px | signal 0.0053
  pos  35 <->  63 | physical   759.9 px | signal 0.0065
  pos  34 <->  67 | physical   790.4 px | signal 0.0068
  pos  19 <->  67 | physical   784.9 px | signal 0.0068


Read the scatter as notebook 57 did: what matters is not just the trend but the **spread**. Points
in the lower-right (far apart, similar signal) are degeneracies. A wide vertical spread at small
physical distance means the map is locally *rough* -- "usually similar, occasionally completely
different" -- which is the signature of positions sitting near a mode's node, where a small move
flips that mode's contribution.

## Summary

In [10]:
print(f'{"representation":48s} {"rho":>8s}')
print('-' * 58)
for k, v in sorted({**res_mag}.items(), key=lambda kv: -kv[1]):
    print(f'{k:48s} {v:+8.4f}')
print()
print('Higher rho = position moves the signal more reliably = more learnable.')
print('Anything at rho~0 is a hash: unlearnable regardless of architecture.')

representation                                        rho
----------------------------------------------------------
log1p(magnitude)                                  +0.3961
sqrt(magnitude)                                   +0.3830
magnitude (current default)                       +0.3519
power (magnitude^2)                               +0.2836
log10(magnitude + eps)                            +0.2278

Higher rho = position moves the signal more reliably = more learnable.
Anything at rho~0 is a hash: unlearnable regardless of architecture.
